# 03b - Validated Multimodal Alignment V2 (Member 2, Phase 2)

Aligns **network windows** (`network_clean.csv`, Member 1) with **radio per-second windows** (`radio_window_features_v2.csv`, Phase 1) using an
**exact one-to-one inner join** on `(scenario_normalized, timestamp_s)`. All logic is in `multimodal_alignment_v2.py` (pandas + NumPy + `zoneinfo`);
this notebook runs it step by step, shows the evidence, writes the outputs and proves that nothing pre-existing changed.

**What this is / is not.** Second-level *temporal* alignment inside one capture. It is **not** packet-to-UE or flow-to-UE correspondence
(no shared packet ID, no UE-to-flow mapping, several UEs pooled per radio window).

**Rules:** no nearest-neighbour or approximate joins · no fabricated rows · no imputation (PUSCH NaNs kept) · Member 1's timezone conversion is *validated*, never repeated or modified ·
labels never decide which records are joined · radio byte-counter deltas are **not** joined (their `(previous, current]` interval needs a separate allocation decision) ·
inputs are exactly the four files named in the task (`network_features.csv`, `radio_clean.csv` and the old `multimodal_dataset.csv` are not inputs).

**Outputs (new files only):** `multimodal_dataset_v2.csv`, `multimodal_feature_columns_v2.json`, `alignment_report_v2.md`, `alignment_report_v2.csv`,
`timezone_validation_v2.csv`, `unmatched_windows_v2.csv`.

Run from the `Multimodal Datasets/` folder (or repo root): *Restart & Run All*, or
`jupyter nbconvert --to notebook --execute --inplace "03b-modality-alignment-v2.ipynb"`.

In [1]:
import sys, hashlib, tempfile, json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

sys.dont_write_bytecode = True
HERE = Path.cwd()
if not (HERE / "multimodal_alignment_v2.py").exists():
    HERE = HERE / "Multimodal Datasets"
assert (HERE / "multimodal_alignment_v2.py").exists(), "Run from the 'Multimodal Datasets' folder or the repo root"
REPO = HERE.parent
sys.path.insert(0, str(HERE))
import multimodal_alignment_v2 as ma

NEW_FILES = {ma.OUT_DATASET, ma.OUT_JSON, ma.OUT_REPORT, ma.OUT_REPORT_CSV, ma.OUT_TZ, ma.OUT_UNMATCHED, ma.OUT_COLLISIONS,
             "03b-modality-alignment-v2.ipynb", "multimodal_alignment_v2.py"}
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()

# Manifest of EVERY pre-existing repository file (taken before anything is written)
PRE_EXISTING = sorted(p for p in REPO.rglob("*") if p.is_file() and p.name not in NEW_FILES
                      and not {"__pycache__", ".git", ".ipynb_checkpoints"} & set(p.parts))
MANIFEST_BEFORE = {str(p.relative_to(REPO)): sha(p) for p in PRE_EXISTING}

print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | {ma.VERSION}")
print("pre-existing repository files hashed:", len(MANIFEST_BEFORE))
for k, pth in [("network_csv", ma.DEFAULT_NETWORK_CSV), ("network_json", ma.DEFAULT_NETWORK_JSON),
               ("radio_csv", ma.DEFAULT_RADIO_CSV), ("radio_json", ma.DEFAULT_RADIO_JSON)]:
    print(f"  input {k:13s} {pth.relative_to(REPO)}")

python 3.12.3 | pandas 3.0.6 | numpy 2.5.4 | multimodal_alignment_v2.0.0
pre-existing repository files hashed: 24
  input network_csv   Network Dataset/network_clean.csv
  input network_json  Network Dataset/network_feature_columns.json
  input radio_csv     Radio Dataset/radio_window_features_v2.csv
  input radio_json    Radio Dataset/radio_feature_columns_v2.json


## Steps 1-2 - Load, validate schemas and key uniqueness, normalise scenario names (`DDOS_` -> `DDOS`)

In [2]:
p = ma.AlignmentPipeline(output_dir=HERE)
p.step1_load_and_validate()
print(f"network_clean: {p.net.shape} | radio windows: {p.radio.shape}")
print(f"network features: {len(p.net_features)} | radio Profile A: {len(p.radio_A)} | radio Profile B: {len(p.radio_B)}")
p.step2_normalize_scenarios()
display(p.checks.frame()[["result", "name", "detail"]])
changed = p.scenario_table[p.scenario_table.radio_scenario != p.scenario_table.network_scenario]
print(f"{len(p.scenario_table)} mappings, {len(changed)} names change, {len(p.scenario_table) - len(changed)} identical; one-to-one validated")
display(changed.T)

network_clean: (4190, 45) | radio windows: (4590, 143)
network features: 29 | radio Profile A: 32 | radio Profile B: 40


,result,name,detail
0,PASS,schema: network_clean.csv has every required c...,"4,190 rows x 45 columns"
1,PASS,schema: radio_window_features_v2.csv has every...,"4,590 rows x 143 columns"
2,PASS,schema: network feature list has no duplicate ...,
3,PASS,schema: Profile A is a subset of Profile B (ra...,
4,PASS,schema: network feature names and radio featur...,overlap=[]
5,PASS,"keys: network (scenario, window_start_epoch_ut...","4,190 unique network windows"
6,PASS,"keys: radio (scenario, timestamp_s) is unique","4,590 unique radio windows"
7,PASS,scenarios: normalisation is injective (no two ...,35 radio names -> 35 distinct
8,PASS,scenarios: every radio scenario maps onto a ne...,radio-only after mapping=[]
9,PASS,scenarios: every network scenario is produced ...,network-only=[]


35 mappings, 12 names change, 23 identical; one-to-one validated


,6,7,8,9,10,11,12,13,14,15,16,17
radio_scenario,DDOS_ICMP1,DDOS_ICMP2,DDOS_ICMP3,DDOS_SYN1,DDOS_SYN2,DDOS_SYN3,DDOS_TCP1,DDOS_TCP2,DDOS_TCP3,DDOS_UDP1,DDOS_UDP2,DDOS_UDP3
network_scenario,DDOSICMP1,DDOSICMP2,DDOSICMP3,DDOSSYN1,DDOSSYN2,DDOSSYN3,DDOSTCP1,DDOSTCP2,DDOSTCP3,DDOSUDP1,DDOSUDP2,DDOSUDP3


## Step 3 - Timestamps: integer-valued epoch seconds on both sides (Member 1's epoch is used as-is)

In [3]:
p.step3_timestamps()
ck = p.checks.frame()
display(ck[ck.name.str.startswith("timestamps")][["result", "name", "detail"]])
print("network join key dtype:", p.net["timestamp_s"].dtype, "| radio join key dtype:", p.radio["timestamp_s"].dtype)
print("network window_start_epoch_utc (source dtype):", p.net["window_start_epoch_utc"].dtype, "-> integer-valued:", bool((p.net["window_start_epoch_utc"] % 1 == 0).all()))

,result,name,detail
14,PASS,timestamps: network window_start_epoch_utc is ...,non-integer values=0
15,PASS,timestamps: network epochs lie in the plausibl...,
16,PASS,timestamps: radio timestamp_s has an integer d...,dtype=int64
17,PASS,timestamps: radio timestamp_s lies in the plau...,
18,PASS,timestamps: radio first/last timestamp_ms fall...,mismatches=0
19,PASS,timestamps: network wall epoch - UTC epoch equ...,
20,PASS,timestamps: window_start_utc text equals the U...,
21,PASS,timestamps: window_id suffix equals 4 x epoch ...,
22,PASS,timestamps: network windows are whole-second a...,


network join key dtype: int64 | radio join key dtype: int64
network window_start_epoch_utc (source dtype): float64 -> integer-valued: True


## Step 4 - Timezone validation against `Europe/Dublin` (`zoneinfo`), all 35 scenarios

In [4]:
p.step4_timezone()
print("Dublin UTC-offset changes in range:", [pd.Timestamp(t, unit="s", tz="UTC").strftime("%Y-%m-%d %H:%M UTC") for t in p.transitions])
print(p.tz["validation_result"].value_counts().to_dict(), "| distinct labels:", sorted(p.tz["recorded_tz_label"].unique()))
display(p.tz)

Dublin UTC-offset changes in range: ['2025-03-30 01:00 UTC']
{'PASS': 35} | distinct labels: ['GMT', 'IST']


,scenario,capture_date_utc,capture_date_dublin,first_window_utc,last_window_utc,n_windows,recorded_tz_label,expected_tz_label_first_window,expected_tz_label_last_window,windows_matching_expected_label,expected_utc_offset_s,applied_offset_s,wall_minus_utc_s,sign_and_offset_consistent,spans_dst_transition,days_to_nearest_dst_transition,validation_result
0,Benign1,2025-04-14,2025-04-14,2025-04-14 12:34:44,2025-04-14 12:36:45,122,IST,IST,IST,122,3600,3600,3600,True,False,15.5,PASS
1,Benign2,2025-04-14,2025-04-14,2025-04-14 12:40:18,2025-04-14 12:45:31,235,IST,IST,IST,235,3600,3600,3600,True,False,15.5,PASS
2,Benign3,2025-04-14,2025-04-14,2025-04-14 12:50:35,2025-04-14 12:55:30,178,IST,IST,IST,178,3600,3600,3600,True,False,15.5,PASS
3,Benign4,2025-04-14,2025-04-14,2025-04-14 12:59:14,2025-04-14 13:04:15,230,IST,IST,IST,230,3600,3600,3600,True,False,15.5,PASS
4,Benign5,2025-04-14,2025-04-14,2025-04-14 13:08:36,2025-04-14 13:11:08,153,IST,IST,IST,153,3600,3600,3600,True,False,15.5,PASS
5,Benign6,2025-04-14,2025-04-14,2025-04-14 13:14:09,2025-04-14 13:19:02,271,IST,IST,IST,271,3600,3600,3600,True,False,15.5,PASS
6,DDOSICMP1,2025-04-03,2025-04-03,2025-04-03 15:20:08,2025-04-03 15:21:48,101,IST,IST,IST,101,3600,3600,3600,True,False,4.6,PASS
7,DDOSICMP2,2025-04-03,2025-04-03,2025-04-03 15:23:07,2025-04-03 15:24:54,108,IST,IST,IST,108,3600,3600,3600,True,False,4.6,PASS
8,DDOSICMP3,2025-04-03,2025-04-03,2025-04-03 15:26:06,2025-04-03 15:27:54,109,IST,IST,IST,109,3600,3600,3600,True,False,4.6,PASS
9,DDOSSYN1,2025-04-03,2025-04-03,2025-04-03 14:39:35,2025-04-03 14:41:15,101,IST,IST,IST,101,3600,3600,3600,True,False,4.6,PASS


## Step 5 - Label consistency (network labels are authoritative) and feature-profile membership

In [5]:
p.step5_labels()
p.validate_profiles()
lt = p.label_table
print({f: int((~lt[f"{f}_match"]).sum()) for f in ["class_fm", "family", "mode", "is_attack", "ddos"]}, "<- mismatching scenarios per field")
print("(radio `ddos` has no network twin; compared with network mode == 'DDoS')")
ck = p.checks.frame()
display(ck[ck.name.str.startswith(("labels", "profiles", "leakage"))][["result", "name", "detail"]])
print("Profile A:", len(p.profile_A), "=", len(p.net_features), "network +", len(p.radio_A), "radio")
print("Profile B:", len(p.profile_B), "=", len(p.net_features), "network +", len(p.radio_B), "radio")

{'class_fm': 0, 'family': 0, 'mode': 0, 'is_attack': 0, 'ddos': 0} <- mismatching scenarios per field
(radio `ddos` has no network twin; compared with network mode == 'DDoS')


,result,name,detail
29,PASS,labels: network labels are constant within eac...,
30,PASS,labels: no missing network labels,"nulls={'family': 0, 'mode': 0, 'class_fm': 0, ..."
31,PASS,labels: radio labels are constant within each ...,
32,PASS,labels: radio class_fm agrees with the network...,mismatching scenarios=[]
33,PASS,labels: radio family agrees with the network s...,mismatching scenarios=[]
34,PASS,labels: radio mode agrees with the network sid...,mismatching scenarios=[]
35,PASS,labels: radio is_attack agrees with the networ...,mismatching scenarios=[]
36,PASS,labels: radio ddos agrees with the network sid...,mismatching scenarios=[]
37,PASS,profiles: network feature count,"expected 29, observed 29"
38,PASS,profiles: radio Profile A column count,"expected 32, observed 32"


Profile A: 61 = 29 network + 32 radio
Profile B: 69 = 29 network + 40 radio


## Step 6 - Exact one-to-one inner join on `(scenario_normalized, timestamp_s)`

In [6]:
p.step6_join()
assert not p.checks.failed(), p.checks.failed()
print(f"network windows {len(p.net):,} | radio windows {len(p.radio):,} | matched {len(p.joined_full):,}")
print(f"network-only {len(p.net_only_keys)} | radio-only {len(p.radio_only_keys)}")
print("sorted by network UTC time:", bool(p.joined_full["timestamp_s"].is_monotonic_increasing))

network windows 4,190 | radio windows 4,590 | matched 4,139
network-only 51 | radio-only 451
sorted by network UTC time: True


## Step 7 - Unmatched observations, per-scenario coverage, zero-packet investigation

In [7]:
p.step7_unmatched()
u = p.unmatched
display(u.groupby(["modality", "boundary_or_interior"]).size().unstack(fill_value=0))
print("position:", u.groupby("modality")["position_relative_to_other_modality"].value_counts().to_dict())
display(p.coverage[["network_scenario", "class_fm", "network_windows", "radio_windows", "matched", "unmatched_network", "unmatched_radio",
                    "network_retention_pct", "radio_retention_pct"]])

boundary_or_interior,boundary,interior
modality,,
network_only,48,3
radio_only,158,293


position: {('network_only', 'after_other_modality_end'): 30, ('network_only', 'before_other_modality_start'): 18, ('network_only', 'inside_other_modality_span'): 3, ('radio_only', 'inside_other_modality_span'): 293, ('radio_only', 'before_other_modality_start'): 116, ('radio_only', 'after_other_modality_end'): 42}


,network_scenario,class_fm,network_windows,radio_windows,matched,unmatched_network,unmatched_radio,network_retention_pct,radio_retention_pct
0,Benign1,Benign,122,128,121,1,7,99.1803,94.5312
1,Benign2,Benign,235,319,235,0,84,100.0000,73.6677
2,Benign3,Benign,178,303,177,1,126,99.4382,58.4158
3,Benign4,Benign,230,305,229,1,76,99.5652,75.0820
4,Benign5,Benign,153,156,152,1,4,99.3464,97.4359
5,Benign6,Benign,271,304,270,1,34,99.6310,88.8158
6,DDOSICMP1,ICMP_DDoS,101,107,101,0,6,100.0000,94.3925
7,DDOSICMP2,ICMP_DDoS,108,115,108,0,7,100.0000,93.9130
8,DDOSICMP3,ICMP_DDoS,109,114,109,0,5,100.0000,95.6140
9,DDOSSYN1,SYN_DDoS,101,106,101,0,5,100.0000,95.2830


In [8]:
z = p.zero_packet
print(f"network_clean zero-packet windows: {z['n_zero_pkt_windows']} (min pkts = {z['min_pkts_in_network_clean']:g}); 01b keeps only pkts > 0")
print(f"seconds absent inside network spans: {z['in_span_absent_total']} -> with a radio window: {z['in_span_absent_with_radio_window']}, without: {z['in_span_absent_without_radio_window']}")
display(z["per_scenario"])
print("supplementary, descriptive only (byte deltas are NOT joined or allocated):", z["radio_volume_overall"])
display(z["radio_volume"])
print("\nWhy an end-second label does not place a delta inside one network window - interval geometry on windows with a valid delta:")
g = p.interval_geometry
print(f"  windows: {g['n_windows_with_delta_interval']:,} | interval starts in the previous second: {100*g['share_start_in_previous_second']:.1f}%")
display(g["fraction_inside_end_second"].round(3).to_frame("fraction of interval inside the end second"))

network_clean zero-packet windows: 0 (min pkts = 2); 01b keeps only pkts > 0
seconds absent inside network spans: 312 -> with a radio window: 293, without: 19


,scenario,absent_seconds_inside_network_span,of_which_with_radio_window,gap_runs,median_run_s,max_run_s
0,Benign2,79,79,69,1.0,4
1,Benign3,118,118,98,1.0,13
2,Benign4,72,72,3,18.0,39
3,Benign6,23,23,13,1.0,4
4,ICMP1,20,1,1,1.0,1


supplementary, descriptive only (byte deltas are NOT joined or allocated): {'n': 293, 'UL_nonzero_share': 1.0, 'DL_nonzero_share': 1.0}


,scenario,n_absent,n_matched,UL_median_absent,UL_median_matched,UL_share_zero_absent,DL_median_absent,DL_median_matched,DL_share_zero_absent
0,Benign2,79,235,455866.0,649737.0,0.0,161514.0,185712.0,0.0
1,Benign3,118,177,397167.0,439550.0,0.0,217698.0,190069.0,0.0
2,Benign4,72,229,27688.0,555681.0,0.0,155791.5,202898.0,0.0
3,Benign6,23,270,32891.0,40461.5,0.0,48835.0,53029.0,0.0
4,ICMP1,1,98,865926.0,837551.0,0.0,20689.0,23281.0,0.0



Why an end-second label does not place a delta inside one network window - interval geometry on windows with a valid delta:
  windows: 4,554 | interval starts in the previous second: 100.0%


,fraction of interval inside the end second
count,4554.000
mean,0.543
std,0.219
min,0.028
5%,0.029
25%,0.464
50%,0.522
75%,0.614
95%,0.932
max,0.989


## Step 8 - Final dataset and metadata (Profile A and Profile B)

In [9]:
p.step8_build_dataset()
ds = p.dataset
print(ds.shape, "| roles:", {k: len(v) for k, v in p.role.items()})
print("identifiers :", p.role["identifiers"]); print("timestamps  :", p.role["timestamps"])
print("labels      :", p.role["labels"]);      print("diagnostics :", p.role["diagnostics"])
print("PUSCH NaN cells kept:", int(ds[[c for c in p.radio_B if c.startswith("puschSnr_")]].isna().sum().sum()))
display(ds.head(3).T.head(30))

(4139, 86) | roles: {'identifiers': 4, 'timestamps': 4, 'labels': 4, 'diagnostics': 5, 'network_features': 29, 'radio_features': 40}
identifiers : ['window_id', 'scenario', 'radio_scenario', 'radio_window_id']
timestamps  : ['timestamp_s', 'window_start_epoch_utc', 'window_start_utc', 't_rel_s']
labels      : ['family', 'mode', 'class_fm', 'is_attack']
diagnostics : ['is_partial', 'coverage_s', 'radio_n_records', 'radio_n_ue_reporting', 'radio_pusch_valid_reports']
PUSCH NaN cells kept: 8


,0,1,2
window_id,PortScan1@6962654200,PortScan1@6962654204,PortScan2@6962654988
scenario,PortScan1,PortScan1,PortScan2
radio_scenario,PortScan1,PortScan1,PortScan2
radio_window_id,PortScan1@1740663550,PortScan1@1740663551,PortScan2@1740663747
timestamp_s,1740663550,1740663551,1740663747
window_start_epoch_utc,1740663550,1740663551,1740663747
window_start_utc,2025-02-27 13:39:10.000000,2025-02-27 13:39:11.000000,2025-02-27 13:42:27.000000
t_rel_s,0.0,1.0,0.0
family,PortScan,PortScan,PortScan
mode,PortScan,PortScan,PortScan


## Step 9 - Integrity, leakage and reproducibility validation (hard assertions must all pass)

In [10]:
p.step9_validate()
res = p.checks.frame()
display(res[["result", "name", "detail"]])
display(p.target_table)
print(f"hard assertions: {int((res.kind == 'assert').sum())} | failed: {len(p.checks.failed())} | target/comparison discrepancies: {len(p.checks.discrepancies())}")
assert not p.checks.failed(), p.checks.failed()

,result,name,detail
0,PASS,schema: network_clean.csv has every required c...,"4,190 rows x 45 columns"
1,PASS,schema: radio_window_features_v2.csv has every...,"4,590 rows x 143 columns"
2,PASS,schema: network feature list has no duplicate ...,
3,PASS,schema: Profile A is a subset of Profile B (ra...,
4,PASS,schema: network feature names and radio featur...,overlap=[]
...,...,...,...
80,PASS,audit target: matched_windows,"expected 4,139, observed 4,139"
81,PASS,audit target: unmatched_network_windows,"expected 51, observed 51"
82,PASS,audit target: unmatched_radio_windows,"expected 451, observed 451"
83,PASS,reproducibility: independent in-memory re-exec...,


,target,expected,observed,status
0,scenario_mappings,35,35,match
1,network_windows,4190,4190,match
2,radio_windows,4590,4590,match
3,matched_windows,4139,4139,match
4,unmatched_network_windows,51,51,match
5,unmatched_radio_windows,451,451,match


hard assertions: 73 | failed: 0 | target/comparison discrepancies: 0


## Write outputs

In [11]:
written = p.write_outputs()
for f in written:
    print(f"{f.name:36s} {f.stat().st_size:>10,d} bytes")
assert all(f.exists() and f.stat().st_size > 0 for f in written)
assert not p.checks.failed(), p.checks.failed()
display(p.checks.frame().tail(2)[["result", "name", "detail"]])

timezone_validation_v2.csv                4,568 bytes
multimodal_dataset_v2.csv             2,344,917 bytes
multimodal_feature_columns_v2.json       14,987 bytes
alignment_report_v2.csv                   5,134 bytes
unmatched_windows_v2.csv                129,647 bytes
alignment_report_v2.md                   29,919 bytes


,result,name,detail
85,PASS,"outputs: dataset CSV round-trips (shape, colum...","4,139 rows x 86 columns"
86,PASS,outputs: unmatched_windows_v2.csv row count eq...,


## Final verification: originals untouched; an independent fresh run reproduces every output byte for byte

In [12]:
MANIFEST_AFTER = {str(x.relative_to(REPO)): sha(x) for x in PRE_EXISTING}
changed = [k for k in MANIFEST_BEFORE if MANIFEST_AFTER.get(k) != MANIFEST_BEFORE[k]]
missing = [k for k in MANIFEST_BEFORE if not (REPO / k).exists()]
print(f"{len(MANIFEST_BEFORE)} pre-existing files checked | changed: {len(changed)} | missing: {len(missing)}")
assert not changed and not missing
for must in ["Network Dataset/network_clean.csv", "Network Dataset/network_feature_columns.json", "Radio Dataset/radio_window_features_v2.csv",
             "Radio Dataset/radio_feature_columns_v2.json", "Multimodal Datasets/multimodal_dataset.csv",
             "Multimodal Datasets/03-modality-alignment.ipynb", "Radio Dataset/radio_clean.csv", "Network Dataset/network_features.csv"]:
    assert must in MANIFEST_BEFORE, must
print("inputs and all protected files present and byte-identical to before the run")

outs = [ma.OUT_DATASET, ma.OUT_JSON, ma.OUT_REPORT, ma.OUT_REPORT_CSV, ma.OUT_TZ, ma.OUT_UNMATCHED]
with tempfile.TemporaryDirectory() as tmp:
    p2 = ma.AlignmentPipeline(output_dir=tmp).run()
    for name in outs:
        same = sha(Path(tmp) / name) == sha(HERE / name)
        print(f"{name:36s} identical to the notebook's output: {same}")
        assert same, name

back = pd.read_csv(HERE / ma.OUT_DATASET); meta = json.load(open(HERE / ma.OUT_JSON))
assert list(back.columns) == meta["exported_columns"]
assert set(meta["profiles"]["B"]["columns"]) <= set(back.columns) and set(meta["profiles"]["A"]["columns"]) <= set(back.columns)
print(f"dataset reloaded: {back.shape[0]:,} rows x {back.shape[1]} columns | Profile A {meta['profiles']['A']['count']} | Profile B {meta['profiles']['B']['count']}")
print("\nALL STEPS COMPLETED SUCCESSFULLY")

24 pre-existing files checked | changed: 0 | missing: 0
inputs and all protected files present and byte-identical to before the run


multimodal_dataset_v2.csv            identical to the notebook's output: True
multimodal_feature_columns_v2.json   identical to the notebook's output: True
alignment_report_v2.md               identical to the notebook's output: True
alignment_report_v2.csv              identical to the notebook's output: True
timezone_validation_v2.csv           identical to the notebook's output: True
unmatched_windows_v2.csv             identical to the notebook's output: True
dataset reloaded: 4,139 rows x 86 columns | Profile A 61 | Profile B 69

ALL STEPS COMPLETED SUCCESSFULLY
